# <font color="#418FDE" size="6.5" uppercase>**Metriken und Suche**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Wählen passende Metriken für Klassifikation und Regression aus. 
- Führen leakage-sichere Kreuzvalidierung mit passenden Splittern durch. 
- Optimieren Hyperparameter mit kleinen Suchräumen und dokumentieren Ergebnisse. 


## **1. Metriken vertiefen**

### **1.1. Klassifikation bewerten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_01_01.jpg?v=1787646703" width="250">



>* Genauigkeit passt vor allem bei ausgeglichenen Klassen
>* Wichtige Fehlerarten hängen vom Anwendungskontext ab

>* Präzision bewertet verlässliche positive Vorhersagen
>* Sensitivität findet Positive, riskiert mehr Fehlalarme

>* Mehrere Metriken fachlich gemeinsam bewerten
>* Fehlerfolgen und Teilgruppenleistung berücksichtigen



In [ ]:
#@title Python-Code - Klassifikation bewerten

# Dieses Beispiel bewertet ein einfaches Klassifikationsmodell.
# Mehrere Metriken zeigen unterschiedliche Fehlerarten.
# Die Konfusionsmatrix macht Fehlentscheidungen sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine kurze Prüfung verhindert unklare Fehlermeldungen später.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung bleibt durch Stratifikation fairer für beide Klassen.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)

# Das Modell lernt aus Trainingsdaten und bewertet Testdaten.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Diese Metriken beantworten verschiedene Bewertungsfragen.
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Genauigkeit: {accuracy:.3f}")
print(f"Präzision: {precision:.3f}")
print(f"Sensitivität: {recall:.3f}")
print(f"F1-Wert: {f1:.3f}")

# Die Matrix zeigt, welche Fehler hinter den Zahlen stehen.
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, colorbar=False)
ax.set_title("Konfusionsmatrix für Testdaten")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Tatsächliche Klasse")
plt.show()



### **1.2. ROC und Precision Recall**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_01_02.jpg?v=1787646707" width="250">



>* Schwellen steuern Treffer und Fehlalarme
>* ROC zeigt Trennung über viele Schwellen

>* Precision bewertet positive Vorhersagen, Recall deren Vollständigkeit
>* Besonders hilfreich bei seltenen positiven Klassen

>* Metrik nach fachlichem Ziel wählen
>* Schwellen anhand praktischer Folgen beurteilen



In [ ]:
#@title Python-Code - ROC und Precision Recall

# Wir vergleichen ROC und Precision-Recall anschaulich.
# Seltene positive Klassen verändern die Metrikperspektive.
# Die Grafik zeigt beide Kurven gemeinsam.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen eine kleine unausgewogene Klassifikationsaufgabe.
features, target = make_classification(
    n_samples=1200,
    n_features=8,
    n_informative=4,
    n_redundant=1,
    weights=[0.92, 0.08],
    class_sep=1.0,
    random_state=42,
)

# Diese Prüfung macht die Klassenverteilung bewusst sichtbar.
positive_rate = target.mean()
if positive_rate <= 0 or positive_rate >= 1:
    raise ValueError("Die Daten brauchen beide Klassen.")

# Stratifikation erhält die seltene positive Klasse im Testset.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# Skalierung wird nur auf Trainingsdaten gelernt.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

# Das Modell liefert Wahrscheinlichkeiten statt fester Klassen.
model.fit(X_train, y_train)
positive_scores = model.predict_proba(X_test)[:, 1]

# Beide Kurven nutzen dieselben Scores, aber andere Blickwinkel.
fpr, tpr, roc_thresholds = roc_curve(y_test, positive_scores)
precision, recall, pr_thresholds = precision_recall_curve(y_test, positive_scores)
roc_auc = auc(fpr, tpr)
pr_auc = auc(recall, precision)

# Eine konkrete Schwelle zeigt den praktischen Zielkonflikt.
threshold = 0.30
predicted_positive = positive_scores >= threshold
true_positive = np.sum((predicted_positive == 1) & (y_test == 1))
false_positive = np.sum((predicted_positive == 1) & (y_test == 0))
false_negative = np.sum((predicted_positive == 0) & (y_test == 1))

# Sichere Nenner vermeiden Division durch null.
precision_at_threshold = true_positive / max(true_positive + false_positive, 1)
recall_at_threshold = true_positive / max(true_positive + false_negative, 1)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Positive Klasse im Testset: {y_test.mean():.1%}")
print(f"ROC-AUC: {roc_auc:.3f}")
print(f"PR-AUC: {pr_auc:.3f}")
print(f"Bei Schwelle {threshold:.2f}: Precision {precision_at_threshold:.3f}")
print(f"Bei Schwelle {threshold:.2f}: Recall {recall_at_threshold:.3f}")

# Eine Achse reicht, weil beide Kurven Werte zwischen null und eins zeigen.
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, label=f"ROC-Kurve, AUC={roc_auc:.3f}")
ax.plot(recall, precision, label=f"Precision-Recall, AUC={pr_auc:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="ROC-Zufallslinie")

ax.set_title("ROC und Precision-Recall bei seltener positiver Klasse")
ax.set_xlabel("ROC: Fehlalarmrate, PR: Recall")
ax.set_ylabel("ROC: Trefferquote, PR: Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.show()



### **1.3. Regressionsfehler verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_01_03.jpg?v=1787646705" width="250">



>* Regressionsfehler messen Abstände zu echten Werten
>* MAE ist verständlich in Originaleinheiten

>* Große Fehler können besonders teuer sein
>* Ausreißer sorgfältig fachlich prüfen

>* Metriken nach Ziel und Fehlerkosten wählen
>* Mehrere Kennzahlen im Anwendungskontext erklären



In [ ]:
#@title Python-Code - Regressionsfehler verstehen

# Dieses Beispiel vergleicht typische Regressionsfehler.
# Große Fehler beeinflussen manche Metriken besonders stark.
# Die Grafik zeigt Fehler in Euro.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import sklearn

# Kleine Beispieldaten zeigen echte und vorhergesagte Mietpreise.
actual_prices = np.array([700, 820, 900, 1000, 1100, 1250], dtype=float)
predicted_prices = np.array([720, 790, 940, 980, 1080, 1500], dtype=float)

# Diese Prüfung verhindert unpassende Arraylängen.
if actual_prices.shape != predicted_prices.shape:
    raise ValueError("Ist- und Vorhersagewerte müssen gleich lang sein.")

# Einzelne Fehler machen die Metriken nachvollziehbar.
errors = predicted_prices - actual_prices
absolute_errors = np.abs(errors)

# MAE bleibt in der ursprünglichen Einheit Euro.
mae = mean_absolute_error(actual_prices, predicted_prices)
rmse = mean_squared_error(actual_prices, predicted_prices) ** 0.5

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"MAE: {mae:.1f} Euro durchschnittliche absolute Abweichung")
print(f"RMSE: {rmse:.1f} Euro, stärker beeinflusst durch große Fehler")
print(f"Größter Einzelfehler: {absolute_errors.max():.0f} Euro")

# Die Balken zeigen, welcher Fall besonders stark abweicht.
case_numbers = np.arange(1, len(actual_prices) + 1)
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(case_numbers, absolute_errors, color="steelblue", label="Absoluter Fehler")
ax.axhline(mae, color="darkorange", linewidth=2, label="MAE")
ax.axhline(rmse, color="crimson", linewidth=2, label="RMSE")

ax.set_title("Regressionsfehler bei Mietpreisvorhersagen")
ax.set_xlabel("Wohnung")
ax.set_ylabel("Fehler in Euro")
ax.legend()

plt.show()



## **2. Leakage sichere Kreuzvalidierung**

### **2.1. KFold Varianten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_02_01.jpg?v=1787646691" width="250">



>* KFold bewertet Modelle über mehrere Folds
>* Datenvorbereitung nur im Trainingsfold lernen

>* Stratifiziere bei ungleichen Klassenverteilungen.
>* Mische nur ohne Zeit- oder Gruppenstruktur.

>* Abhängigkeiten und Einsatzsituation bestimmen den Split
>* Fold-Anzahl passend zur Datenmenge wählen



In [ ]:
#@title Python-Code - KFold Varianten

# Wir vergleichen KFold-Varianten an einem Klassifikationsdatensatz.
# Stratifizierung hält Klassenanteile in jedem Fold stabil.
# Pipelines verhindern Datenleckage bei der Skalierung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Ein unausgewogener Datensatz macht Fold-Unterschiede gut sichtbar.
features, target = make_classification(
    n_samples=300,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    weights=[0.9, 0.1],
    random_state=42,
)

# Diese Prüfung schützt vor unerwarteten Datenformen.
if features.shape[0] != target.shape[0]:
    raise ValueError("Merkmale und Zielwerte haben unterschiedlich viele Zeilen.")

# Die Pipeline skaliert nur innerhalb des jeweiligen Trainingsfolds.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42),
)

# KFold mischt zufällig, beachtet aber keine Klassenanteile.
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Wir messen die positive Klasse in jedem Validierungsfold.
kfold_rates = []
stratified_rates = []
for _, test_index in kfold.split(features):
    kfold_rates.append(float(np.mean(target[test_index])))

for _, test_index in stratified.split(features, target):
    stratified_rates.append(float(np.mean(target[test_index])))

# Kreuzvalidierung nutzt dieselben Splitter für faire Modellbewertung.
kfold_scores = cross_val_score(model, features, target, cv=kfold, scoring="f1")
stratified_scores = cross_val_score(
    model,
    features,
    target,
    cv=stratified,
    scoring="f1",
)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Gesamter Anteil positiver Klasse: {np.mean(target):.2f}")
print(f"KFold positive Anteile: {np.round(kfold_rates, 2).tolist()}")
print(f"StratifiedKFold positive Anteile: {np.round(stratified_rates, 2).tolist()}")
print(f"Mittlerer F1 mit KFold: {np.mean(kfold_scores):.2f}")
print(f"Mittlerer F1 mit StratifiedKFold: {np.mean(stratified_scores):.2f}")

# Das Diagramm zeigt, wie stabil die Klassenanteile bleiben.
fold_numbers = np.arange(1, 6)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fold_numbers, kfold_rates, marker="o", label="KFold")
ax.plot(fold_numbers, stratified_rates, marker="o", label="StratifiedKFold")

ax.axhline(np.mean(target), color="gray", linestyle="--", label="Gesamtanteil")
ax.set_title("Klassenanteil pro Validierungsfold")
ax.set_xlabel("Fold")
ax.set_ylabel("Anteil positiver Klasse")
ax.set_xticks(fold_numbers)

ax.set_ylim(0, 0.25)
ax.legend()
plt.tight_layout()
plt.show()



### **2.2. Gruppen und Zeit**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_02_02.jpg?v=1787646696" width="250">



>* Gruppen nie zufällig über Falten verteilen
>* Validierung soll neue Gruppen realistisch testen

>* Gruppen nach späterer Vorhersageeinheit bilden
>* Ehrliche Schätzung trotz ungleicher Gruppengrößen

>* Zeitliche Reihenfolge bei Validierung strikt einhalten
>* Nur damals verfügbare Merkmale verwenden



In [ ]:
#@title Python-Code - Gruppen und Zeit

# Wir vergleichen zufällige und gruppierte Kreuzvalidierung.
# Gruppen dürfen nicht zwischen Falten wandern.
# Die Ausgabe zeigt realistischere Validierungswerte.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine Messdaten mit wiederholten Gruppen.
rng = np.random.default_rng(42)
n_groups = 30
rows_per_group = 6

# Jede Gruppe hat ein eigenes Muster und ein eigenes Ziel.
groups = np.repeat(np.arange(n_groups), rows_per_group)
group_signal = rng.normal(size=n_groups)
y = (group_signal > 0).astype(int)

# Die Zeilen enthalten ein starkes gruppenspezifisches Merkmal.
row_noise = rng.normal(scale=0.25, size=n_groups * rows_per_group)
feature_one = group_signal[groups] + row_noise
feature_two = rng.normal(size=n_groups * rows_per_group)

# Das Modell sieht nur Merkmale, nicht die Gruppennummer.
X = np.column_stack([feature_one, feature_two])
y_rows = y[groups]

# Diese Prüfung macht die Gruppengröße explizit sichtbar.
if len(X) != len(y_rows) or len(groups) != len(y_rows):
    raise ValueError("X, y und groups müssen gleich lang sein.")

# Eine Pipeline verhindert Leakage durch Skalierung vor der Faltung.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=200),
)

# Zufällige Falten mischen Zeilen derselben Gruppe oft zusammen.
random_cv = KFold(n_splits=5, shuffle=True, random_state=42)
random_scores = cross_val_score(model, X, y_rows, cv=random_cv, scoring="accuracy")

# GroupKFold hält alle Zeilen einer Gruppe zusammen.
group_cv = GroupKFold(n_splits=5)
group_scores = cross_val_score(
    model,
    X,
    y_rows,
    cv=group_cv,
    groups=groups,
    scoring="accuracy",
)

# Wir zählen, ob Gruppen in zufälligen Falten geteilt werden.
random_split = next(random_cv.split(X, y_rows))
train_groups = set(groups[random_split[0]])
valid_groups = set(groups[random_split[1]])
shared_groups = len(train_groups.intersection(valid_groups))

# Die Ausgabe bleibt kurz und fokussiert.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Geteilte Gruppen in einer zufälligen Falte: {shared_groups}")
print(f"Zufällige CV Genauigkeit: {random_scores.mean():.2f}")
print(f"GroupKFold Genauigkeit: {group_scores.mean():.2f}")

# Das Balkendiagramm macht den Leakage-Effekt sichtbar.
fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Zufällige CV", "GroupKFold"]
means = [random_scores.mean(), group_scores.mean()]

ax.bar(labels, means, color=["tab:orange", "tab:blue"])
ax.set_ylim(0, 1)
ax.set_ylabel("Mittlere Genauigkeit")
ax.set_title("Gruppen-Leakage verändert die Bewertung")

plt.show()



### **2.3. Mehrere Metriken prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_02_03.jpg?v=1787646692" width="250">



>* Mehrere Metriken zeigen unterschiedliche Fehlerarten
>* Validierungsdaten strikt vor Leakage schützen

>* Metriken zeigen Zielkonflikte und Fehlerarten
>* Folds zeigen Stabilität über Datenaufteilungen

>* Hauptmetrik vorab festlegen, Diagnosemetriken trennen
>* Streuung und Einbrüche je Split prüfen



In [ ]:
#@title Python-Code - Mehrere Metriken prüfen

# Wir prüfen mehrere Metriken in Kreuzvalidierung.
# Die Pipeline verhindert Leakage bei der Skalierung.
# Die Ausgabe zeigt Mittelwert und Streuung.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung macht die Beispielannahmen sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# StratifiedKFold erhält die Klassenanteile in jedem Fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Die Skalierung wird innerhalb jedes Trainingsfolds gelernt.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Mehrere Metriken werden auf denselben Validierungsfolds berechnet.
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

# Cross_validate liefert pro Metrik einen Wert je Fold.
results = cross_validate(model, X, y, cv=cv, scoring=scoring)

# Wir fassen Mittelwert und Streuung übersichtlich zusammen.
metric_names = ["accuracy", "precision", "recall", "f1"]
summary_rows = []

for metric_name in metric_names:
    scores = results["test_" + metric_name]
    summary_rows.append([metric_name, scores.mean(), scores.std()])

summary = pd.DataFrame(
    summary_rows,
    columns=["Metrik", "Mittelwert", "Streuung"]
)

# Die Tabelle bleibt klein und gut lesbar.
print("scikit-learn Version:", sklearn.__version__)
print(summary.round(3).to_string(index=False))
print("Wichtig: Alle Metriken nutzen dieselben leakage-sicheren Folds.")

# Das Balkendiagramm vergleicht die Metriken visuell.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary["Metrik"], summary["Mittelwert"], yerr=summary["Streuung"])

ax.set_title("Mehrere Metriken in leakage-sicherer Kreuzvalidierung")
ax.set_xlabel("Metrik")
ax.set_ylabel("Validierungswert")
ax.set_ylim(0.85, 1.0)

plt.show()



## **3. Hyperparameter gezielt suchen**

### **3.1. Rastersuche mit GridSearchCV**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_03_01.jpg?v=1787646698" width="250">



>* GridSearchCV prüft alle gewählten Hyperparameter-Kombinationen.
>* Gut für kleine, begründete Suchräume.

>* Mehrere Validierungen machen die Auswahl robuster
>* Passende Metriken steuern die Optimierung

>* Kleine, begründete Suchräume statt Werteflut
>* Pipeline nutzen und Ergebnisse kritisch dokumentieren



In [ ]:
#@title Python-Code - Rastersuche mit GridSearchCV

# Dieses Beispiel zeigt eine kleine Rastersuche.
# GridSearchCV prüft mehrere Hyperparameter mit Kreuzvalidierung.
# Die Ausgabe dokumentiert die beste Modellwahl.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung verhindert unklare Datenprobleme.
if X.shape[0] != y.shape[0] or X.shape[0] < 100:
    raise ValueError("Die Datenform passt nicht zum Beispiel.")

# Der Testteil bleibt bis zur finalen Bewertung unberührt.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Die Pipeline skaliert nur innerhalb der jeweiligen Trainingsfalte.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", KNeighborsClassifier())]
)

# Der Suchraum ist bewusst klein und gut nachvollziehbar.
param_grid = {
    "model__n_neighbors": [3, 5, 9],
    "model__weights": ["uniform", "distance"],
}

# StratifiedKFold erhält ähnliche Klassenanteile in jeder Falte.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV bewertet jede Kombination mit derselben Metrik.
grid_search = GridSearchCV(
    pipeline, param_grid, scoring="f1", cv=cv, n_jobs=1
)

# Ein einziger Fit startet die komplette Rastersuche.
grid_search.fit(X_train, y_train)

# Die besten Einstellungen werden auf dem Testteil geprüft.
y_pred = grid_search.predict(X_test)
test_f1 = f1_score(y_test, y_pred)

# Eine kleine Tabelle zeigt die geprüften Kombinationen.
results = pd.DataFrame(grid_search.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
plot_data = results.head(5).copy()

# Kurze Ausgaben dokumentieren Version, Auswahl und Testleistung.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Beste Parameter: {grid_search.best_params_}")
print(f"F1 im Testdatensatz: {test_f1:.3f}")

# Die Grafik macht die besten Kreuzvalidierungswerte vergleichbar.
labels = []
for _, row in plot_data.iterrows():
    label = f"k={row['param_model__n_neighbors']}, {row['param_model__weights']}"
    labels.append(label)

# Genau eine Achse reicht für diesen Vergleich aus.
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, plot_data["mean_test_score"], color="steelblue")
ax.set_title("Beste GridSearchCV-Kombinationen nach F1")
ax.set_xlabel("Hyperparameterkombination")
ax.set_ylabel("Mittlerer F1-Wert in der Kreuzvalidierung")
ax.set_ylim(0.85, 1.0)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()



### **3.2. Zufällige Suche**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_03_02.jpg?v=1787646700" width="250">



>* Zufällige Kombinationen statt vollständiger Rastersuche
>* Rechenbudget breiter und oft effizienter nutzen

>* Ideal für kontinuierliche, unsichere Hyperparameterbereiche
>* Begrenzte Zufallsversuche fair per Kreuzvalidierung bewerten

>* Suchräume realistisch begrenzen und Budget beachten
>* Zufall fixieren und Ergebnisse nachvollziehbar dokumentieren



In [ ]:
#@title Python-Code - Zufällige Suche

# Wir demonstrieren zufällige Suche für Hyperparameter.
# Kreuzvalidierung bewertet jede gezogene Modellkonfiguration fair.
# Die Ausgabe zeigt beste Werte und Suchmuster.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

# Wir laden einen kleinen Klassifikationsdatensatz aus scikit-learn.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung macht die Datengröße bewusst sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der Splitter erhält die Klassenverteilung in jeder Falte.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(random_state=42, n_jobs=-1)

# Der Suchraum bleibt klein und gut nachvollziehbar.
param_distributions = {
    "n_estimators": [50, 80, 120, 160],
    "max_depth": [2, 3, 4, 5, None],
    "min_samples_leaf": [1, 2, 4, 8],
}

# RandomizedSearchCV zieht nur einige Kombinationen zufällig.
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)

# Eine einzige Suche trainiert mehrere kleine Modelle intern.
search.fit(X, y)
results = pd.DataFrame(search.cv_results_)

# Wir sortieren die getesteten Konfigurationen nach Validierungsleistung.
summary = results.sort_values("mean_test_score", ascending=False)
summary = summary.head(5)

# Die Tabelle dokumentiert die wichtigsten Suchergebnisse kompakt.
shown_columns = [
    "mean_test_score",
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_leaf",
]

# Kurze Ausgaben halten Metrik, Version und Ergebnis fest.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Beste Kreuzvalidierungs-Accuracy: {search.best_score_:.3f}")
print(f"Beste Hyperparameter: {search.best_params_}")

# Die Grafik zeigt, wie stark die Versuche schwanken.
plot_scores = results["mean_test_score"].to_numpy()
trial_numbers = np.arange(1, len(plot_scores) + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(trial_numbers, plot_scores, label="gezogene Konfiguration")
ax.axhline(search.best_score_, color="orange", label="bestes Ergebnis")
ax.set_title("Zufällige Suche mit festem Budget")
ax.set_xlabel("Versuch")
ax.set_ylabel("Mittlere Accuracy in Kreuzvalidierung")
ax.legend()
plt.show()



### **3.3. Suchergebnisse dokumentieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_03_03.jpg?v=1787646701" width="250">



>* Suchweg und Entscheidungen nachvollziehbar festhalten
>* Metrik, Splits und Leakage sauber dokumentieren

>* Suchstruktur und Ergebnisbreite nachvollziehbar festhalten
>* Stabilität, Einfachheit und Aufwand mitbewerten

>* Validierung und Testdaten klar trennen
>* Auswahl, Begründung und Grenzen dokumentieren



In [ ]:
#@title Python-Code - Suchergebnisse dokumentieren

# Wir dokumentieren eine kleine Hyperparametersuche nachvollziehbar.
# Kreuzvalidierung liefert vergleichbare Ergebnisse für Kandidaten.
# Am Ende entsteht eine kompakte Ergebnisübersicht.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Wir laden einen kleinen Klassifikationsdatensatz aus scikit-learn.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung macht die Datengrundlage explizit.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Pipeline verhindert Datenleckage bei der Skalierung.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)

# Der Suchraum bleibt klein und fachlich gut dokumentierbar.
param_grid = {"model__C": [0.01, 0.1, 1.0, 10.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV speichert alle Kandidaten und ihre Validierungswerte.
search = GridSearchCV(
    pipeline, param_grid, scoring="f1", cv=cv, return_train_score=False
)
search.fit(X, y)

# Wir formen die wichtigsten Suchergebnisse als kleine Tabelle.
results = pd.DataFrame(search.cv_results_)
summary = results[["param_model__C", "mean_test_score", "std_test_score"]].copy()
summary.columns = ["C", "mittlerer_F1", "streuung_F1"]

# Sortieren zeigt die besten Kandidaten zuerst.
summary = summary.sort_values("mittlerer_F1", ascending=False)
summary["mittlerer_F1"] = summary["mittlerer_F1"].round(3)
summary["streuung_F1"] = summary["streuung_F1"].round(3)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Metrik: F1, Splitter: StratifiedKFold mit {cv.n_splits} Folds")
print(f"Gewähltes C: {search.best_params_['model__C']}")
print(summary.head(4).to_string(index=False))

# Die Grafik dokumentiert Leistung und Streuung der Kandidaten.
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(summary["C"], summary["mittlerer_F1"], yerr=summary["streuung_F1"], fmt="o-")
ax.set_xscale("log")
ax.set_title("Dokumentierte Hyperparametersuche")
ax.set_xlabel("Regularisierung C")
ax.set_ylabel("Mittlerer F1-Wert")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Metriken und Suche**</font>


In this lecture, you learned to:
- Wählen passende Metriken für Klassifikation und Regression aus. 
- Führen leakage-sichere Kreuzvalidierung mit passenden Splittern durch. 
- Optimieren Hyperparameter mit kleinen Suchräumen und dokumentieren Ergebnisse. 

In the next Lecture (Lecture B), we will go over 'Merkmale erklären'